# Round 5 Manual — Ashflow Alpha Portfolio

Budget B = 1,000,000 XIREC. 9 Ignith products. Hold to next day.

**Fee formula:** `fee_i = (v_i/100)^2 * B` where `v_i` is % allocation. With B=1M → `fee_i = 100 * p_i^2` (p in %).

**Per-product PnL** (r is net return, p is alloc in %):
$$\text{PnL}_i = 10{,}000 \cdot r_i \cdot p_i - 100 \cdot p_i^2$$

**Constraints:** $\sum |p_i| \leq 100$, $p_i \in \mathbb{Z}$, sign(p_i) = direction (long if +, short if −).

**Continuous optimum (unconstrained):** $p_i^* = 50 r_i$ (i.e. $|r|/2$ in fraction = $50r$ in %).

**Used budget subtracted from PnL.** Unused expires worthless. Better than overcommit since fee is convex per product.

Inspired by prev-year Iceberg writeup. Same recipe:
1. Tag each product with sentiment.
2. Map sentiment → expected return.
3. Solve convex problem with cvxpy (continuous).
4. Round to integer %, verify budget.

In [49]:
import numpy as np
import cvxpy as cp
import itertools

## Sentiment per product

Tradable goods (9 from submission panel): Obsidian cutlery, Pyroflex cells, Thermalite core, Lava cake, Magma ink, Scoria paste, Ashes of the Phoenix, Volcanic incense, Sulfur reactor.

Read each Ashflow Alpha article, assign sentiment. Magnitudes debatable — judgment call.

| Product | Article | Direction | Magnitude rationale |
|---|---|---|---|
| Obsidian cutlery | factory halt, contamination, industry implications | -- | scandal + halt, reputation hit dominates scarcity |
| Pyroflex cells | 50% tax cut killed, levy doubles, slows purchases | --- | concrete quantitative hit, abrupt |
| Thermalite core | users 1.42M → 3.89M (2.7×), 16h42m daily | ++ | hard numbers, structural growth |
| Lava cake | actual lava found, sales halted, lawsuits | ---- | strongest bear: halt + lawsuits |
| Magma ink | sold inside Lava Fountain Pen hot drop, 6h queue, post-merger | + | demand spike but widely promoted — priced in |
| Scoria paste | Lava D. Ray pump on stream | + | influencer pump, crowd will buy → priced near range top |
| Ashes of the Phoenix | PR scandal (phoenix burned), immortal-bird defense | - | sentiment hit but no actual supply issue |
| Volcanic incense | Whiff Nostralico pump | + | pump, similar to Scoria |
| Sulfur reactor | added to Elemental Index 118 | ++ | mechanical index-fund buying |

In [ ]:
sentiments = {
    'ObsidianCutlery':   '-',
    'PyroflexCells':     '---',
    'ThermaliteCore':    '++++',
    'LavaCake':          '----',
    'MagmaInk':          '+',
    'ScoriaPaste':       '+', 
    'AshesOfThePhoenix': '=',
    'VolcanicIncense':   '+', 
    'SulfurReactor':     '+++',
}

returns_map = {
    '=': 0.00,
    '+':    0.05,
    '++':   0.15,
    '+++':  0.25,
    '++++': 0.35,
    '-':   -0.05,
    '--':  -0.10,
    '---': -0.30,
    '----':-0.60,
}


sentiments_map = {
     'ObsidianCutlery':   -0.05,
    'PyroflexCells':     -0.30,
    'ThermaliteCore':    0.35,
    'LavaCake':          '----',
    'MagmaInk':          '+',
    'ScoriaPaste':       '+', 
    'AshesOfThePhoenix': 0.01,
    'VolcanicIncense':   '+', 
    'SulfurReactor':     '+++',

}


products = list(sentiments.keys())
rets = np.array([returns_map[sentiments[p]] for p in products])

for p, r in zip(products, rets):
    print(f'{p:20s} {sentiments[p]:5s} r={r:+.2f}')

ObsidianCutlery      -     r=-0.05
PyroflexCells        ---   r=-0.30
ThermaliteCore       ++++  r=+0.35
LavaCake             ----  r=-0.60
MagmaInk             +     r=+0.05
ScoriaPaste          +     r=+0.05
AshesOfThePhoenix    =     r=+0.00
VolcanicIncense      +     r=+0.05
SulfurReactor        +++   r=+0.25


## Continuous optimum via cvxpy

Maximize $\sum_i (10000 \cdot r_i \cdot p_i - 100 \cdot p_i^2)$ s.t. $\sum_i |p_i| \leq 100$.

Equivalent to minimize $100 \cdot \|p\|_2^2 - 10000 \cdot r^T p$.

In [51]:
B = 1_000_000
n = len(products)

p = cp.Variable(n)
objective = cp.Minimize(100 * cp.sum_squares(p) - 10_000 * rets @ p)
constraints = [cp.norm(p, 1) <= 100]
prob = cp.Problem(objective, constraints)
prob.solve()

p_cont = p.value

print('Continuous optimal allocation:')
for prod, pi in zip(products, p_cont):
    print(f'  {prod:20s} {pi:+7.3f}%')
print(f'\nL1 used: {np.sum(np.abs(p_cont)):.2f}%')
print(f'Expected PnL: {-prob.value:,.2f} XIREC')

Polishing not needed - no active set detected at optimal point
Continuous optimal allocation:
  ObsidianCutlery       -2.500%
  PyroflexCells        -15.000%
  ThermaliteCore       +17.500%
  LavaCake             -30.000%
  MagmaInk              +2.500%
  ScoriaPaste           +2.500%
  AshesOfThePhoenix     -0.000%
  VolcanicIncense       +2.500%
  SulfurReactor        +12.500%

L1 used: 85.00%
Expected PnL: 161,250.00 XIREC


## Integer-constrained solution

Continuous optimum should match $p_i^* = 50 r_i$ when budget not binding. Round to nearest integer.
If $\sum |p_i^*| < 100$ strictly, rounding is the integer optimum (objective separable + convex per coord).

In [52]:
def pnl(p_vec, r_vec):
    return float(np.sum(10_000 * r_vec * p_vec - 100 * p_vec**2))

p_int = np.round(p_cont).astype(int)

# if budget binding, do greedy fixup
while np.sum(np.abs(p_int)) > 100:
    # remove unit from coord with smallest marginal benefit
    margins = []
    for i in range(n):
        s = np.sign(p_int[i]) if p_int[i] != 0 else 0
        if s == 0: 
            margins.append(np.inf)
            continue
        # PnL change if we shrink |p_i| by 1
        new_p = p_int[i] - s
        d = (10_000*rets[i]*new_p - 100*new_p**2) - (10_000*rets[i]*p_int[i] - 100*p_int[i]**2)
        margins.append(-d)  # cost of shrinking
    i_drop = int(np.argmin(margins))
    p_int[i_drop] -= int(np.sign(p_int[i_drop]))

print('Integer allocation:')
for prod, pi in zip(products, p_int):
    side = 'BUY' if pi > 0 else ('SELL' if pi < 0 else '---')
    cost = abs(pi) * B // 100
    fee = int(100 * pi**2)
    print(f'  {prod:20s} {side:4s} {abs(pi):3d}%  cost={cost:>9,}  fee={fee:>7,}')

print(f'\nTotal L1 used: {np.sum(np.abs(p_int))}%')
print(f'Total fees:    {int(np.sum(100 * p_int**2)):,}')
print(f'Expected PnL:  {pnl(p_int, rets):,.0f} XIREC')

Integer allocation:
  ObsidianCutlery      SELL   3%  cost=   30,000  fee=    900
  PyroflexCells        SELL  15%  cost=  150,000  fee= 22,500
  ThermaliteCore       BUY   18%  cost=  180,000  fee= 32,400
  LavaCake             SELL  30%  cost=  300,000  fee= 90,000
  MagmaInk             BUY    3%  cost=   30,000  fee=    900
  ScoriaPaste          BUY    3%  cost=   30,000  fee=    900
  AshesOfThePhoenix    ---    0%  cost=        0  fee=      0
  VolcanicIncense      BUY    3%  cost=   30,000  fee=    900
  SulfurReactor        BUY   13%  cost=  130,000  fee= 16,900

Total L1 used: 88%
Total fees:    165,400
Expected PnL:  161,100 XIREC


## Sensitivity — vary sentiment magnitudes

Returns are guesses. Check robustness by perturbing the magnitude scale.

In [53]:
def solve_for_returns(r_vec):
    p = cp.Variable(n)
    obj = cp.Minimize(100 * cp.sum_squares(p) - 10_000 * r_vec @ p)
    cons = [cp.norm(p, 1) <= 100]
    prob = cp.Problem(obj, cons)
    prob.solve()
    p_int = np.round(p.value).astype(int)
    while np.sum(np.abs(p_int)) > 100:
        margins = []
        for i in range(len(r_vec)):
            s = np.sign(p_int[i]) if p_int[i] != 0 else 0
            if s == 0:
                margins.append(np.inf); continue
            new_p = p_int[i] - s
            d = (10_000*r_vec[i]*new_p - 100*new_p**2) - (10_000*r_vec[i]*p_int[i] - 100*p_int[i]**2)
            margins.append(-d)
        i_drop = int(np.argmin(margins))
        p_int[i_drop] -= int(np.sign(p_int[i_drop]))
    return p_int, pnl(p_int, r_vec)

scales = [0.5, 0.75, 1.0, 1.25, 1.5]
print(f'{"scale":>6s} | {"PnL":>10s} | allocs')
for s in scales:
    pi, val = solve_for_returns(s * rets)
    sig = ' '.join(f'{x:+d}' for x in pi)
    print(f'{s:6.2f} | {val:>10,.0f} | {sig}')

 scale |        PnL | allocs
Polishing not needed - no active set detected at optimal point
  0.50 |     40,250 | -1 -8 +9 -15 +1 +1 +0 +1 +6
Polishing not needed - no active set detected at optimal point
  0.75 |     90,650 | -2 -11 +13 -23 +2 +2 +0 +2 +9
Polishing not needed - no active set detected at optimal point
  1.00 |    161,100 | -3 -15 +18 -30 +3 +3 +0 +3 +13
  1.25 |    251,250 | -2 -18 +21 -37 +2 +2 +0 +2 +15
  1.50 |    352,600 | +0 -19 +23 -42 +0 +0 +0 +0 +15


## Submission table

In [54]:
print(f'{"Product":20s} {"Side":5s} {"Pct":>5s} {"Cost":>10s} {"Fee":>8s}')
print('-' * 56)
for prod, pi in zip(products, p_int):
    if pi == 0:
        continue
    side = 'Buy' if pi > 0 else 'Sell'
    pct = abs(int(pi))
    cost = pct * B // 100
    fee = 100 * pct**2
    print(f'{prod:20s} {side:5s} {pct:>4d}% {cost:>10,} {fee:>8,}')
print('-' * 56)
print(f'{"Total":20s} {" ":5s} {np.sum(np.abs(p_int)):>4d}% {np.sum(np.abs(p_int))*B//100:>10,} {int(np.sum(100*p_int**2)):>8,}')
print(f'\nExpected PnL: {pnl(p_int, rets):,.0f} XIREC (= profit on top of used budget)')

Product              Side    Pct       Cost      Fee
--------------------------------------------------------
ObsidianCutlery      Sell     3%     30,000      900
PyroflexCells        Sell    15%    150,000   22,500
ThermaliteCore       Buy     18%    180,000   32,400
LavaCake             Sell    30%    300,000   90,000
MagmaInk             Buy      3%     30,000      900
ScoriaPaste          Buy      3%     30,000      900
VolcanicIncense      Buy      3%     30,000      900
SulfurReactor        Buy     13%    130,000   16,900
--------------------------------------------------------
Total                        88%    880,000  165,400

Expected PnL: 161,100 XIREC (= profit on top of used budget)
